In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.axes import Axes
from matplotlib.figure import Figure
from scipy import stats
import statsmodels.api as sm

from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    brier_score_loss,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [3]:
#loading the data
euro_test_df: pd.DataFrame = pd.read_csv("../data/EuroTest.csv")
euro_training_df: pd.DataFrame = pd.read_csv("../data/EuroTraining.csv")
handball_long_df: pd.DataFrame = pd.read_csv("../data/handball_long.csv")
handball_wide_df: pd.DataFrame = pd.read_csv("../data/handball_wide.csv")
tennis_df: pd.DataFrame = pd.read_csv("../data/SlamsData.csv")

In [4]:
import pandas as pd
df = euro_training_df
print("=== DATASET DIMENSIONS ===")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\n=== COLUMN NAMES ===")
print(df.columns.tolist())

print("\n=== FIRST 5 ROWS ===")
print(df.head().to_string())

print("\n=== DATA TYPES ===")
print(df.dtypes.to_string())

print("\n=== MISSING VALUES ===")
print(df.isnull().sum().to_string())

print("\n=== DUPLICATED ROWS ===")
print(df.duplicated().sum())

print("\n=== UNIQUE VALUES PER COLUMN ===")
print(df.nunique().to_string())

print("\n=== NUMERICAL STATISTICS ===")
print(df.describe().to_string())

print("\n=== CATEGORICAL VARIABLES ===")
for col in df.select_dtypes(include=["object", "category"]).columns:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(10).to_string())

=== DATASET DIMENSIONS ===
Rows: 390
Columns: 10

=== COLUMN NAMES ===
['Goals', 'Team', 'Opponent', 'Year', 'GroupStage', 'GDP', 'MarketValue', 'FifaRank', 'UefaPoints', 'CLPlayers']

=== FIRST 5 ROWS ===
   Goals            Team        Opponent  Year  GroupStage       GDP  MarketValue  FifaRank  UefaPoints  CLPlayers
0      0     Switzerland  Czech Republic  2008           1  1.155086    -0.313333        42      -10700       -1.0
1      1  Czech Republic     Switzerland  2008           1 -1.155086     0.313333       -42       10700        1.0
2      2        Portugal          Turkey  2008           1  0.834200     0.936769       -16       10042        5.0
3      0          Turkey        Portugal  2008           1 -0.834200    -0.936769        16      -10042       -5.0
4      1  Czech Republic        Portugal  2008           1 -0.134616    -0.888390        -3       -9258       -4.0

=== DATA TYPES ===
Goals            int64
Team               str
Opponent           str
Year           

C:\Users\loubia\AppData\Local\Temp\ipykernel_27052\1936023503.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object", "category"]).columns:


# Tennis Data

## Logistic regression

### Splitting of the data :
**football data:** its already split training and testing datasetes

**tennis data:** following sheet 1 we can use split the data from 2011
to 2017 serve as training data and the data from 2018 to 2024 serve as test data.

In [ ]:
#Change to max 3 numbers after the comma
cols_to_round = ["elo", "prob", "age.30"]
tennis_df[cols_to_round] = tennis_df[cols_to_round].round(3)
# Remove the last 2 columns in-place from tennis_df (they are not that useful)
#tennis_df = tennis_df.iloc[:, :-2]

In [ ]:
print(tennis_df.columns)

delete the variable sets because those informations are available only after tha game, for testing it doesnt make sense to know the sets but not the result so they are taken off.

the the CV folds we dont use the usual folds because the data is chronogically bound. We have to respect that and a solution would be the walk-forward validation

In [ ]:
columns_to_drop = ["sets.1", "sets.2","B365.1", "B365.2","player.1", "player.2","surface","AvgProb.1", "AvgProb.2","prob"]
#deleted surface because it is redundant with the tournament.
#deleted prob because we dont work with probs
#deleted average because they re redundant
tennis_df = tennis_df.drop(columns=columns_to_drop)

In [ ]:
print(tennis_df["year"].value_counts())

In [ ]:
# Training data: 2011 to 2021
tennis_train_df: pd.DataFrame = tennis_df[tennis_df["year"].between(2011, 2021)].copy()

# Test data: 2022 to 2024
tennis_test_df: pd.DataFrame = tennis_df[tennis_df["year"].between(2022, 2024)].copy()

generally a 80/20 split for training/testing is a good idea. so by taking the last 3 years as testing data corresponds well. For the rest 11cv folds for each year is actually a good idea.

In [ ]:
# Training data
X_train = tennis_train_df.drop(columns=["win", "year"])
y_train = tennis_train_df["win"]

# Test data
X_test = tennis_test_df[X_train.columns]
y_test = tennis_test_df["win"]

# Keep years for cross-validation
years_train = tennis_train_df["year"]

In [ ]:
# Select predictors
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = ["tournament"]

X_train_model = X_train[numeric_cols + categorical_cols]
X_test_model = X_test[numeric_cols + categorical_cols]

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"),
     categorical_cols)
])

# Ridge logistic regression
model = Pipeline([
    ("preprocessor", preprocessor),
    ("logistic", LogisticRegression(max_iter=1000))
])

### This ensures that, during cross-validation:
1. The scaler learns the mean and standard deviation from the current training fold.
2. It transforms that training fold.
3. The logistic regression is trained.
4. The validation fold is transformed using the same scaler, without recalculating its parameters.

In [ ]:
#generating the folds:
cv_folds = []

for val_year in range(2015, 2022):

    train_idx = np.where(years_train.to_numpy() < val_year)[0]
    val_idx = np.where(years_train.to_numpy() == val_year)[0]

    cv_folds.append((train_idx, val_idx))

## small pause from the model to check the features

In [ ]:
model.fit(X_train, y_train)

In [ ]:
# Get feature names after preprocessing
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

# Get logistic regression coefficients
coefficients = model.named_steps["logistic"].coef_[0]

# Create feature importance table
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Importance": np.abs(coefficients)
})

# Sort by absolute coefficient
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance.round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.barh(
    feature_importance["Feature"],
    feature_importance["Coefficient"]
)

plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Standardized coefficient")
plt.title("Ridge Logistic Regression Coefficients")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "log_loss": "neg_log_loss",
    "brier": "neg_brier_score"
}

cv_results = cross_validate(
    model,
    X_train_model,
    y_train,
    cv=cv_folds,
    scoring=scoring,
    return_train_score=True
)

In [ ]:
results_df = pd.DataFrame({
    "Year": range(2015, 2022),
    "Train Accuracy": cv_results["train_accuracy"],
    "Validation Accuracy": cv_results["test_accuracy"],
    "Balanced Accuracy": cv_results["test_balanced_accuracy"],
    "Precision": cv_results["test_precision"],
    "Recall": cv_results["test_recall"],
    "F1": cv_results["test_f1"],
    "ROC-AUC": cv_results["test_roc_auc"],
    "Log Loss": -cv_results["test_log_loss"],
    "Brier Score": -cv_results["test_brier"]
})

print(results_df.round(4).to_string(index=False))
mean_results = results_df.drop(columns="Year").mean()

print("\nAverage Cross-Validation Performance:")
print(mean_results.round(4))

In [ ]:
#Fit the model on all the training now and test on the test data that never see
model.fit(X_train_model, y_train)
# Predicted winners (0 or 1)
y_pred = model.predict(X_test_model)

# Predicted probability of winning (class 1)
y_prob = model.predict_proba(X_test_model)[:, 1]
#we need to import these metrics now because in the cross validation it is already implemented in the methods but now we are using them directly
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    brier_score_loss,
    confusion_matrix
)

test_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC",
        "Log Loss",
        "Brier Score"
    ],
    "Test": [
        accuracy_score(y_test, y_pred),
        balanced_accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob),
        log_loss(y_test, y_prob),
        brier_score_loss(y_test, y_prob)
    ]
})

print(test_results.round(4).to_string(index=False))
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Loss", "Win"],
    cmap="Blues"
)

plt.title("Confusion Matrix - Test Set (2022–2024)")
plt.show()

# UEFA Data

In [ ]:
features = [
    "GroupStage",
    "GDP",
    "MarketValue",
    "FifaRank",
    "UefaPoints",
    "CLPlayers"
]

target = "Goals"

# Training data
X_train = euro_training_df[features]
y_train = euro_training_df[target]

# Test data
X_test = euro_test_df[features]
y_test = euro_test_df[target]

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

In [ ]:
# Find the year column
year_col = next(
    (col for col in euro_training_df.columns
     if col.lower() == "year"),
    None
)

if year_col is None:
    raise ValueError("Please check the name of your year column.")

years_train = euro_training_df[year_col].to_numpy()
unique_years = sorted(np.unique(years_train))

cv_folds = []
validation_years = []

for val_year in unique_years[2:]:
    train_idx = np.where(years_train < val_year)[0]
    val_idx = np.where(years_train == val_year)[0]

    cv_folds.append((train_idx, val_idx))
    validation_years.append(val_year)

print("Validation years:", validation_years)

In [ ]:
from sklearn.model_selection import cross_validate

scoring = {
    "mae": "neg_mean_absolute_error",
    "mse": "neg_mean_squared_error",
    "r2": "r2",
    "medae": "neg_median_absolute_error"
}

cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv_folds,
    scoring=scoring,
    return_train_score=True
)

In [ ]:
results_df = pd.DataFrame({
    "Year": validation_years,
    "Train MAE": -cv_results["train_mae"],
    "Validation MAE": -cv_results["test_mae"],
    "Validation MSE": -cv_results["test_mse"],
    "Validation RMSE": np.sqrt(-cv_results["test_mse"]),
    "Validation R2": cv_results["test_r2"],
    "Validation Median AE": -cv_results["test_medae"]
})

print(results_df.round(4).to_string(index=False))

In [ ]:
mean_results = results_df.drop(columns="Year").mean()

print("\nAverage Cross-Validation Performance:")
print(mean_results.round(4))

In [ ]:
model.fit(X_train, y_train)

In [ ]:
coefficients = model.named_steps["ridge"].coef_

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": coefficients,
    "Importance": np.abs(coefficients)
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance.round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.barh(
    feature_importance["Feature"],
    feature_importance["Coefficient"]
)

plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Standardized coefficient")
plt.title("UEFA - Ridge Regression Coefficients")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import get_scorer

test_results = {}

for metric_name, scorer_name in scoring.items():
    scorer = get_scorer(scorer_name)

    test_results[metric_name] = scorer(
        model, X_test, y_test
    )

# Convert negative error scores to positive values
test_results["mae"] *= -1
test_results["mse"] *= -1
test_results["medae"] *= -1

# Calculate RMSE
test_results["rmse"] = np.sqrt(test_results["mse"])

print("\nTest Performance:")
print(pd.Series(test_results).round(4))

In [ ]:
comparison = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2", "Median AE"],

    "Cross-validation": [
        results_df["Validation MAE"].mean(),
        results_df["Validation MSE"].mean(),
        results_df["Validation RMSE"].mean(),
        results_df["Validation R2"].mean(),
        results_df["Validation Median AE"].mean()
    ],

    "Test": [
        test_results["mae"],
        test_results["mse"],
        test_results["rmse"],
        test_results["r2"],
        test_results["medae"]
    ]
})

print("Results for the Logistic regression")
print(comparison.round(4).to_string(index=False))

# Going back to Tennis Dataset

## Random forest

In [ ]:
# Data preparation and Random Forest pipeline

In [ ]:
numeric_cols_rf = ["rank", "points", "elo", "age.30"]
categorical_cols_rf = ["tournament"]

features_rf = numeric_cols_rf + categorical_cols_rf

tennis_train_df = tennis_df[tennis_df["year"].between(2011, 2021)].copy()
tennis_test_df = tennis_df[tennis_df["year"].between(2022, 2024)].copy()

X_train_rf = tennis_train_df[features_rf].copy()
y_train_rf = tennis_train_df["win"]

X_test_rf = tennis_test_df[features_rf].copy()
y_test_rf = tennis_test_df["win"]

years_train_rf = tennis_train_df["year"]

rf_preprocessor = ColumnTransformer([
    ("num", "passthrough", numeric_cols_rf),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols_rf)
])

rf_pipeline = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("forest", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=1
    ))
])


In [ ]:
# Expanding-window cross-validation folds

In [ ]:

rf_cv_folds = []
rf_validation_years = []

years_rf = years_train_rf.to_numpy()

for val_year in range(2015, 2022):

    train_idx = np.where(years_rf < val_year)[0]
    val_idx = np.where(years_rf == val_year)[0]

    rf_cv_folds.append((train_idx, val_idx))
    rf_validation_years.append(val_year)

print("Validation years:", rf_validation_years)


In [ ]:
#Hyperparameter tuning with GridSearchCV

In [1]:

rf_param_grid = {
    "forest__ccp_alpha": [0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1],
    "forest__min_samples_split": [20, 30, 40, 45, 55, 60],
    "forest__max_depth": [2, 3, 5, 7, 10, 13, 15]
}

rf_grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring="neg_log_loss",
    cv=rf_cv_folds,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
    refit=True
)

rf_grid_search.fit(X_train_rf, y_train_rf)

print("Best hyperparameters:")
print(rf_grid_search.best_params_)

print("\nBest CV Log Loss:")
print(round(-rf_grid_search.best_score_, 4))


NameError: name 'GridSearchCV' is not defined

In [ ]:
# Examine the hyperparameter tuning results

In [ ]:

rf_grid_results = pd.DataFrame(
    rf_grid_search.cv_results_
)

rf_grid_summary = rf_grid_results[[
    "param_forest__ccp_alpha",
    "param_forest__min_samples_split",
    "param_forest__max_depth",
    "mean_train_score",
    "mean_test_score",
    "rank_test_score"
]].copy()

rf_grid_summary = rf_grid_summary.rename(columns={
    "param_forest__ccp_alpha": "ccp_alpha",
    "param_forest__min_samples_split": "minsplit",
    "param_forest__max_depth": "maxdepth",
    "mean_train_score": "Train Log Loss",
    "mean_test_score": "Validation Log Loss",
    "rank_test_score": "Rank"
})

rf_grid_summary["Train Log Loss"] *= -1
rf_grid_summary["Validation Log Loss"] *= -1

rf_grid_summary = rf_grid_summary.sort_values("Rank")

print(rf_grid_summary.head(10).round(4).to_string(index=False))


In [ ]:
# Cross-validation performance with eight metrics

In [ ]:

rf_best_model = rf_grid_search.best_estimator_

rf_scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "log_loss": "neg_log_loss",
    "brier": "neg_brier_score"
}

rf_cv_results = cross_validate(
    rf_best_model,
    X_train_rf,
    y_train_rf,
    cv=rf_cv_folds,
    scoring=rf_scoring,
    return_train_score=True,
    n_jobs=-1
)

rf_results_df = pd.DataFrame({
    "Year": rf_validation_years,
    "Train Accuracy": rf_cv_results["train_accuracy"],
    "Validation Accuracy": rf_cv_results["test_accuracy"],
    "Balanced Accuracy": rf_cv_results["test_balanced_accuracy"],
    "Precision": rf_cv_results["test_precision"],
    "Recall": rf_cv_results["test_recall"],
    "F1": rf_cv_results["test_f1"],
    "ROC-AUC": rf_cv_results["test_roc_auc"],
    "Log Loss": -rf_cv_results["test_log_loss"],
    "Brier Score": -rf_cv_results["test_brier"]
})

print(rf_results_df.round(4).to_string(index=False))

rf_mean_results = rf_results_df.drop(columns="Year").mean()

print("\nAverage Cross-Validation Performance:")
print(rf_mean_results.round(4))


In [ ]:
# Random Forest feature importance

In [ ]:

rf_feature_names = (
    rf_best_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

rf_importances = (
    rf_best_model.named_steps["forest"]
    .feature_importances_
)

rf_feature_importance = pd.DataFrame({
    "Feature": rf_feature_names,
    "Importance": rf_importances
})

rf_feature_importance = rf_feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(rf_feature_importance.round(4).to_string(index=False))

plt.figure(figsize=(9, 5))

plt.barh(
    rf_feature_importance["Feature"],
    rf_feature_importance["Importance"]
)

plt.xlabel("Feature Importance")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


In [ ]:
# Final evaluation on the test set (2022–2024)

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    brier_score_loss
)

rf_y_pred = rf_best_model.predict(X_test_rf)

rf_y_prob = rf_best_model.predict_proba(X_test_rf)[:, 1]

rf_test_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC",
        "Log Loss",
        "Brier Score"
    ],
    "Test": [
        accuracy_score(y_test_rf, rf_y_pred),
        balanced_accuracy_score(y_test_rf, rf_y_pred),
        precision_score(y_test_rf, rf_y_pred, zero_division=0),
        recall_score(y_test_rf, rf_y_pred),
        f1_score(y_test_rf, rf_y_pred, zero_division=0),
        roc_auc_score(y_test_rf, rf_y_prob),
        log_loss(y_test_rf, rf_y_prob),
        brier_score_loss(y_test_rf, rf_y_prob)
    ]
})

print(rf_test_results.round(4).to_string(index=False))


In [ ]:
# confusio matrix

In [ ]:

from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test_rf,
    rf_y_pred,
    display_labels=["Loss", "Win"],
    cmap="Blues"
)

plt.title("Random Forest Confusion Matrix - Test Set (2022–2024)")
plt.show()


## XGBOOST

In [ ]:
# Imports, existing data, and pipeline

In [ ]:
# Reuse the existing tennis data and folds
X_tennis_train = X_train_rf
y_tennis_train = y_train_rf

X_tennis_test = X_test_rf
y_tennis_test = y_test_rf

tennis_cv_folds = rf_cv_folds
tennis_validation_years = rf_validation_years

# XGBoost pipeline
xgb_pipeline = Pipeline([
    ("preprocessor", clone(rf_preprocessor)),
    ("xgb", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        importance_type="gain",
        random_state=42,
        n_jobs=1
    ))
])


In [ ]:
# Hyperparameter tuning using GridSearchCV

In [ ]:

xgb_param_grid = {
    "xgb__n_estimators": [100, 300],
    "xgb__max_depth": [2, 4, 6],
    "xgb__learning_rate": [0.03, 0.1],
    "xgb__min_child_weight": [1, 5],
    "xgb__subsample": [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0]
}

xgb_grid_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    scoring="neg_log_loss",
    cv=tennis_cv_folds,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
    refit=True
)

xgb_grid_search.fit(X_tennis_train, y_tennis_train)

print("Best hyperparameters:")
print(xgb_grid_search.best_params_)

print("\nBest CV Log Loss:")
print(round(-xgb_grid_search.best_score_, 4))


In [ ]:
# Examine the tuning results

In [ ]:

xgb_grid_results = pd.DataFrame(
    xgb_grid_search.cv_results_
)

xgb_grid_summary = xgb_grid_results[[
    "param_xgb__n_estimators",
    "param_xgb__max_depth",
    "param_xgb__learning_rate",
    "param_xgb__min_child_weight",
    "param_xgb__subsample",
    "param_xgb__colsample_bytree",
    "mean_train_score",
    "mean_test_score",
    "rank_test_score"
]].copy()

xgb_grid_summary = xgb_grid_summary.rename(columns={
    "param_xgb__n_estimators": "n_estimators",
    "param_xgb__max_depth": "max_depth",
    "param_xgb__learning_rate": "learning_rate",
    "param_xgb__min_child_weight": "min_child_weight",
    "param_xgb__subsample": "subsample",
    "param_xgb__colsample_bytree": "colsample_bytree",
    "mean_train_score": "Train Log Loss",
    "mean_test_score": "Validation Log Loss",
    "rank_test_score": "Rank"
})

xgb_grid_summary["Train Log Loss"] *= -1
xgb_grid_summary["Validation Log Loss"] *= -1

xgb_grid_summary = xgb_grid_summary.sort_values("Rank")

print(xgb_grid_summary.head(10).round(4).to_string(index=False))


In [ ]:
# Expanding-window CV with eight metrics

In [ ]:

xgb_best_model = xgb_grid_search.best_estimator_

xgb_scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "log_loss": "neg_log_loss",
    "brier": "neg_brier_score"
}

xgb_cv_results = cross_validate(
    xgb_best_model,
    X_tennis_train,
    y_tennis_train,
    cv=tennis_cv_folds,
    scoring=xgb_scoring,
    return_train_score=True,
    n_jobs=-1
)

xgb_results_df = pd.DataFrame({
    "Year": tennis_validation_years,
    "Train Accuracy": xgb_cv_results["train_accuracy"],
    "Validation Accuracy": xgb_cv_results["test_accuracy"],
    "Balanced Accuracy": xgb_cv_results["test_balanced_accuracy"],
    "Precision": xgb_cv_results["test_precision"],
    "Recall": xgb_cv_results["test_recall"],
    "F1": xgb_cv_results["test_f1"],
    "ROC-AUC": xgb_cv_results["test_roc_auc"],
    "Log Loss": -xgb_cv_results["test_log_loss"],
    "Brier Score": -xgb_cv_results["test_brier"]
})

print(xgb_results_df.round(4).to_string(index=False))

xgb_mean_results = xgb_results_df.drop(columns="Year").mean()

print("\nAverage Cross-Validation Performance:")
print(xgb_mean_results.round(4))


In [ ]:
# XGBoost feature importance

In [ ]:

# Get feature names after preprocessing
xgb_feature_names = (
    xgb_best_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Get XGBoost feature importance
xgb_importances = (
    xgb_best_model.named_steps["xgb"]
    .feature_importances_
)

# Create feature importance table
xgb_feature_importance = pd.DataFrame({
    "Feature": xgb_feature_names,
    "Importance": xgb_importances
})

# Sort by importance
xgb_feature_importance = xgb_feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(xgb_feature_importance.round(4).to_string(index=False))

# Visualize feature importance
plt.figure(figsize=(9, 5))

plt.barh(
    xgb_feature_importance["Feature"],
    xgb_feature_importance["Importance"]
)

plt.xlabel("Feature Importance (Gain)")
plt.title("XGBoost Feature Importance")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


In [ ]:
# Final evaluation on the test set (2022–2024)

In [ ]:


xgb_y_pred = xgb_best_model.predict(X_tennis_test)

xgb_y_prob = xgb_best_model.predict_proba(X_tennis_test)[:, 1]

xgb_test_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC",
        "Log Loss",
        "Brier Score"
    ],
    "Test": [
        accuracy_score(y_tennis_test, xgb_y_pred),
        balanced_accuracy_score(y_tennis_test, xgb_y_pred),
        precision_score(y_tennis_test, xgb_y_pred, zero_division=0),
        recall_score(y_tennis_test, xgb_y_pred),
        f1_score(y_tennis_test, xgb_y_pred, zero_division=0),
        roc_auc_score(y_tennis_test, xgb_y_prob),
        log_loss(y_tennis_test, xgb_y_prob),
        brier_score_loss(y_tennis_test, xgb_y_prob)
    ]
})

print(xgb_test_results.round(4).to_string(index=False))


In [ ]:
# Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_tennis_test,
    xgb_y_pred,
    display_labels=["Loss", "Win"],
    cmap="Blues"
)

plt.title("XGBoost Confusion Matrix - Test Set (2022–2024)")
plt.show()

## Neural network

In [ ]:
# Neural Network pipeline

In [ ]:
nn_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols_rf),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols_rf)
])

nn_pipeline = Pipeline([
    ("preprocessor", nn_preprocessor),
    ("mlp", MLPClassifier(
        solver="lbfgs",
        max_iter=8000,
        random_state=42
    ))
])

In [ ]:
# Hyperparameter tuning using GridSearchCV

In [ ]:

nn_param_grid = {
    "mlp__hidden_layer_sizes": [(3,), (5,), (10,), (5, 5)],
    "mlp__activation": ["tanh", "relu"],
    "mlp__alpha": [0.01, 0.1, 1.0]
}

nn_grid_search = GridSearchCV(
    estimator=nn_pipeline,
    param_grid=nn_param_grid,
    scoring="neg_log_loss",
    cv=rf_cv_folds,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
    refit=True
)

nn_grid_search.fit(X_train_rf, y_train_rf)

print("Best hyperparameters:")
print(nn_grid_search.best_params_)

print("\nBest CV Log Loss:")
print(round(-nn_grid_search.best_score_, 4))

print("\nBest Neural Network:")
print(nn_grid_search.best_estimator_)


In [ ]:
# Examine hyperparameter tuning results

In [ ]:

nn_grid_results = pd.DataFrame(
    nn_grid_search.cv_results_
)

nn_grid_summary = nn_grid_results[[
    "param_mlp__hidden_layer_sizes",
    "param_mlp__activation",
    "param_mlp__alpha",
    "mean_train_score",
    "mean_test_score",
    "rank_test_score"
]].copy()

nn_grid_summary = nn_grid_summary.rename(columns={
    "param_mlp__hidden_layer_sizes": "Hidden Layers",
    "param_mlp__activation": "Activation",
    "param_mlp__alpha": "Alpha",
    "mean_train_score": "Train Log Loss",
    "mean_test_score": "Validation Log Loss",
    "rank_test_score": "Rank"
})

nn_grid_summary["Train Log Loss"] *= -1
nn_grid_summary["Validation Log Loss"] *= -1

nn_grid_summary = nn_grid_summary.sort_values("Rank")

print(nn_grid_summary.head(10).round(4).to_string(index=False))


In [ ]:
# Expanding-window CV with eight metrics

In [ ]:

nn_best_model = nn_grid_search.best_estimator_

nn_cv_results = cross_validate(
    nn_best_model,
    X_train_rf,
    y_train_rf,
    cv=rf_cv_folds,
    scoring=rf_scoring,
    return_train_score=True,
    n_jobs=-1
)

nn_results_df = pd.DataFrame({
    "Year": rf_validation_years,
    "Train Accuracy": nn_cv_results["train_accuracy"],
    "Validation Accuracy": nn_cv_results["test_accuracy"],
    "Balanced Accuracy": nn_cv_results["test_balanced_accuracy"],
    "Precision": nn_cv_results["test_precision"],
    "Recall": nn_cv_results["test_recall"],
    "F1": nn_cv_results["test_f1"],
    "ROC-AUC": nn_cv_results["test_roc_auc"],
    "Log Loss": -nn_cv_results["test_log_loss"],
    "Brier Score": -nn_cv_results["test_brier"]
})

print(nn_results_df.round(4).to_string(index=False))

nn_mean_results = nn_results_df.drop(columns="Year").mean()

print("\nAverage Cross-Validation Performance:")
print(nn_mean_results.round(4))


In [ ]:
# Neural Network feature importance

In [ ]:
nn_importance_train_idx, nn_importance_val_idx = rf_cv_folds[-1]

nn_importance_model = clone(nn_best_model)

nn_importance_model.fit(
    X_train_rf.iloc[nn_importance_train_idx],
    y_train_rf.iloc[nn_importance_train_idx]
)

nn_permutation = permutation_importance(
    nn_importance_model,
    X_train_rf.iloc[nn_importance_val_idx],
    y_train_rf.iloc[nn_importance_val_idx],
    scoring="neg_log_loss",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

nn_feature_importance = pd.DataFrame({
    "Feature": X_train_rf.columns,
    "Importance": nn_permutation.importances_mean,
    "Std": nn_permutation.importances_std
})

nn_feature_importance = nn_feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(nn_feature_importance.round(4).to_string(index=False))

plt.figure(figsize=(9, 5))

plt.barh(
    nn_feature_importance["Feature"],
    nn_feature_importance["Importance"],
    xerr=nn_feature_importance["Std"]
)

plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Increase in Log Loss when feature is shuffled")
plt.title("Neural Network Feature Importance (2021 Validation)")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


In [ ]:
# Final test evaluation (2022–2024)

In [ ]:
nn_y_pred = nn_best_model.predict(X_test_rf)

nn_y_prob = nn_best_model.predict_proba(X_test_rf)[:, 1]

nn_test_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC",
        "Log Loss",
        "Brier Score"
    ],
    "Test": [
        accuracy_score(y_test_rf, nn_y_pred),
        balanced_accuracy_score(y_test_rf, nn_y_pred),
        precision_score(y_test_rf, nn_y_pred, zero_division=0),
        recall_score(y_test_rf, nn_y_pred),
        f1_score(y_test_rf, nn_y_pred, zero_division=0),
        roc_auc_score(y_test_rf, nn_y_prob),
        log_loss(y_test_rf, nn_y_prob),
        brier_score_loss(y_test_rf, nn_y_prob)
    ]
})

print(nn_test_results.round(4).to_string(index=False))


In [ ]:
# Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test_rf,
    nn_y_pred,
    display_labels=["Loss", "Win"],
    cmap="Blues"
)

plt.title("Neural Network Confusion Matrix - Test Set (2022–2024)")
plt.show()

# test

In [11]:

import requests
import pandas as pd
from getpass import getpass

API_KEY = getpass("Enter your Zafronix API key: ")

BASE_URL = "https://api.zafronix.com/uefa/euro/v1"
HEADERS = {"X-API-Key": API_KEY}

years = [2004, 2008, 2012, 2016, 2020, 2024]

all_matches = []

for year in years:
    response = requests.get(
        f"{BASE_URL}/matches",
        headers=HEADERS,
        params={"year": year},
        timeout=30
    )

    response.raise_for_status()
    payload = response.json()

    matches = payload.get("data")

    if not isinstance(matches, list):
        raise ValueError(
            f"Unexpected API response for {year}: "
            f"{list(payload.keys())}"
        )

    df_year = pd.json_normalize(matches, sep="_")
    df_year["Year"] = year

    all_matches.append(df_year)

    print(f"Euro {year}: {len(df_year)} matches")

euro_matches = pd.concat(
    all_matches, ignore_index=True
)

euro_matches.to_csv(
    "Zafronix_Euro_Matches.csv",
    index=False
)

print("Saved:", euro_matches.shape)
print(euro_matches.columns.tolist())


KeyboardInterrupt: Interrupted by user

In [12]:
euro_matches.to_csv(
    "Zafronix_Euro_Matches.csv",
    index=False
)